# 01 Exploratory data analysis

Purpose: understand the data before any modelling. This notebook checks
missingness by vintage, the mix of loan terms over time, and how borrower
features have shifted between the training and out-of-time samples.

Inputs: `data/processed/loans_labelled.parquet` (created by `python -m src.data_prep`).
Excluded loans (2007 to 2011) are left out of every table here.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config
from src.splits import assign_sample, sample_summary
from src.eda import missingness_by_year, term_mix, bad_rate_by_term_year, feature_psi, numeric_profile
from src.data_prep import label_summary

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

df = assign_sample(pd.read_parquet(config.PROCESSED_PATH))
modelling = df[df["sample"] != "excluded"].copy()
print(f"Labelled loans: {len(df):,}  |  in modelling samples: {len(modelling):,}")

## 1. Missingness by issue year

A feature that is mostly missing in the training years cannot be used for
the scorecard without a special bin, and recent-only fields must be checked
for coverage in each vintage. Features above 30% missing in the training
sample are flagged for a decision.

In [ ]:
CANDIDATES = [
    "dti", "fico_range_low", "revol_util", "revol_bal", "open_acc", "total_acc",
    "mths_since_last_delinq", "mort_acc", "collections_12_mths_ex_med",
    "annual_inc", "emp_length", "inq_last_6mths", "pub_rec", "delinq_2yrs",
]
miss = missingness_by_year(modelling, CANDIDATES)
miss = miss.loc[:, sorted(miss.columns)]
display(miss.round(3))

train_miss = modelling.loc[modelling["sample"] == "train", CANDIDATES].isna().mean()
flagged = train_miss[train_miss > 0.30]
print("Flagged (above 30% missing in training):", list(flagged.index) or "none")

## 2. Loan term mix by issue year

Lending Club's 60-month loans issued in 2016 and 2017 are mostly still
Current, so they were removed by the final-outcome rule. The table shows
how much the term mix shifts, and whether it could explain part of the
rise in the 12-month bad rate.

In [ ]:
mix = term_mix(modelling)
display(mix.round(3))

In [ ]:
by_term = bad_rate_by_term_year(modelling)
bad_pivot = by_term.pivot(index="year", columns="term", values="bad_rate")
loan_pivot = by_term.pivot(index="year", columns="term", values="loans")
display(bad_pivot.round(4))
display(loan_pivot)

Read this table as: within each term, does the bad rate still rise in
2016 to 2017? If it does, the shift is not only a term-mix effect.

## 3. Feature shift: training against out-of-time

PSI on each numeric feature, computed on the training bins and applied to
the out-of-time sample. Rule of thumb: below 0.10 stable, 0.10 to 0.25
moderate shift, above 0.25 major shift. These results feed the feature
selection decisions in Stage 3.

In [ ]:
# Numeric features only. emp_length is text ("8 years", "< 1 year") and is
# converted to years in Stage 3, so it is excluded here.
NUMERIC = [c for c in CANDIDATES if c in modelling.columns and pd.api.types.is_numeric_dtype(modelling[c])]
print("Numeric features used:", NUMERIC)
train_s = modelling[modelling["sample"] == "train"]
oot_s = modelling[modelling["sample"] == "oot"]

psi_table = feature_psi(train_s, oot_s, NUMERIC)
display(psi_table.round(4))

In [ ]:
profile = pd.concat(
    {
        "train": numeric_profile(train_s, NUMERIC)["median"],
        "oot": numeric_profile(oot_s, NUMERIC)["median"],
    },
    axis=1,
)
display(profile.round(2))

Medians show direction; PSI shows how much the distribution has moved.
Features with large PSI and large median change are the first candidates
for a root-cause explanation in Stage 3.